# 02. Baseline Retrieval — Qualitative Check

Before quantitative sweeps, we eyeball retrieval on a small set of probe
queries to confirm:

1. The collection is reachable and returns sensible hits.
2. The cosine-score range we'll be sweeping over is realistic.
3. An adversarial / out-of-domain query returns nothing (or very low scores).

This notebook deliberately uses **`retrieve_simple`** (pure cosine,
no query expansion, no severity / applicability boosts) so we observe
the raw embedding signal — the boosts are studied separately in notebook 05.


In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
from src.eval_utils import resolve_eval_paths

CORPUS_DIR, QUERIES_PATH, COLLECTION = resolve_eval_paths(ROOT)
os.environ["QDRANT_URL"] = "http://localhost:6433"
os.environ["AGENT_KMS_COLLECTION"] = COLLECTION
os.environ["AGENT_KMS_CONFIG"] = str(ROOT / ".agent-kms" / "kms.toml")
os.environ["AGENT_KMS_KNOWLEDGE_ROOT"] = str(CORPUS_DIR)

from agent_kms.retrieve import retrieve_simple


In [ ]:
probe_queries = [
    # in-domain, direct
    "useStateでstate変数を追加する",
    # in-domain, paraphrased
    "親から渡された値を子コンポーネントで初期値に使ったあと書き換えたい",
    # JS, technical
    "try catch で例外を捕捉する書き方",
    # JS, intent
    "配列を変更せずに新しい配列を返したい",
    # adversarial — out-of-domain
    "PostgreSQL の logical replication を設定する手順",
]

for q in probe_queries:
    print(f"\n──── {q!r}")
    hits = retrieve_simple(q, limit=5)
    for h in hits:
        src = h["source"].split("/")[-1]
        print(f"  {h['score']:.3f}  {src:45s}  | {h['heading'][:50]}")


## Observations

- **In-domain queries top out around cosine ≈ 0.87** for `multilingual-e5-base`.
  This is well **below the agent-kms default threshold of 0.93** — on this
  corpus, the default would silently return zero results for most queries.
- The Japanese-on-Japanese baseline cosine is high (~0.80 floor) even for
  unrelated content, so the *discrimination* between relevant and noise
  is in a narrow 0.05-0.08 band, not a 0.10+ band.
- The out-of-domain query (PostgreSQL) returns hits at ~0.78-0.80 — i.e.
  the model **does** confidently assign lower scores to irrelevant content,
  but that lower band overlaps with the on-topic band of harder paraphrased
  queries. The threshold sweep in notebook 03 quantifies this trade-off.
